# Getting Started with Prompt Engineering

This notebook contains examples and exercises to learning about prompt engineering.

We will be using the [Google AI Studio](https://aistudio.google.com/) for all examples.

---

## 1. Prompt Engineering Basics

Objectives
- Load the libraries
- Review the format
- Cover basic prompts
- Review common use cases

Below we are loading the necessary libraries, utilities, and configurations.

In [ ]:
%%capture
# update or install the necessary libraries
!pip install --upgrade google-genai

In [ ]:
import IPython
from google import genai
from google.genai import types
from google.colab import userdata

Load environment variables. You should create your own free Gemini API key in [Google AI Studio](https://aistudio.google.com/), Store the key in Colab Secrets(the 🔑 icon in the Colab left sidebar) as `GEMINI_API_KEY`. Do not paste any API key directly into the notebook.

In [ ]:
# API configuration
gemini_api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=gemini_api_key)

In [ ]:
MODEL_NAME = "gemini-3.1-flash-lite"


def set_open_params(
    model=MODEL_NAME,
    temperature=0.7,
    max_tokens=1024,
    top_p=1,
):
    """Bundle Gemini generation parameters into a dict."""
    return {
        "model": model,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "top_p": top_p,
    }


def get_completion(params, prompt):
    """Send a prompt to Gemini and return the generated text."""
    config = types.GenerateContentConfig(
        temperature=params["temperature"],
        top_p=params["top_p"],
        max_output_tokens=params["max_tokens"],
    )
    response = client.models.generate_content(
        model=params["model"],
        contents=prompt,
        config=config,
    )
    return (response.text or "").strip()


Basic prompt example:

In [ ]:
# basic example
params = set_open_params()

prompt = "The sky is"

generated_text = get_completion(params, prompt)

IPython.display.Markdown(generated_text)

Try different temperature or top-p to compare results, and why do they differ or not?



In [ ]:
params = set_open_params(temperature=0)
prompt = "The sky is"
response = get_completion(params, prompt)
IPython.display.Markdown(response)

In [ ]:
params = set_open_params(top_p=0)
prompt = "The sky is"
response = get_completion(params, prompt)
IPython.display.Markdown(response)

<details>
<summary><b>Aside: how <code>temperature</code> and <code>top_p</code> actually shape the output</b> (click to expand)</summary>

This is a side note. It is useful for tuning the decoding side, but not the focus of this notebook. Skim or skip as you prefer. You can also use the *Text Summarization* prompt from §1.1 below to help you learn this part :)

Both parameters change the next-token distribution before sampling. They act on different parts of it.

**Temperature** rescales the whole distribution. At each step the model produces logits over the vocabulary. We divide the logits by `T` and apply softmax again. When `T < 1` the distribution gets sharper. The top tokens grab even more probability mass. When `T > 1` the distribution gets flatter. Low-probability tokens become viable. At `T = 0` the distribution collapses to a single point on the argmax. This is pure greedy decoding. At `T = 1` the original distribution is used as is.

**Top_p** (nucleus sampling) truncates the distribution instead. We sort tokens by probability. We keep the smallest set whose cumulative probability reaches `p`. The rest are discarded. The survivors are then renormalised. With `top_p = 1` nothing is cut. With `top_p = 0` only the argmax survives. This is also pure greedy decoding. Values like `0.9` or `0.95` chop off the long tail of unlikely tokens. The relative weights among the top tokens stay the same.

This explains what you saw earlier. `temperature = 0` and `top_p = 0` are two different ways to reach greedy decoding. So they give the same output. They behave differently at intermediate values. Temperature can revive unlikely tokens by flattening the curve. Top_p cannot. Once a token is truncated it is gone.

A reasonable default: leave one at its identity value (`temperature = 1` or `top_p = 1`) and tune the other. Setting both is fine, but the interaction is harder to reason about. On modern instruction-tuned models the gain over tuning just one is usually small.

For a longer treatment of decoding strategies, including top-k, min-p, and typical sampling, see Hugging Face's [*How to generate text*](https://huggingface.co/blog/how-to-generate) blog post.

</details>

### 1.1 Text Summarization

In [ ]:
params = set_open_params(temperature=0.7)
prompt = """Antibiotics are a type of medication used to treat bacterial infections. They work by either killing the bacteria or preventing them from reproducing, allowing the body's immune system to fight off the infection. Antibiotics are usually taken orally in the form of pills, capsules, or liquid solutions, or sometimes administered intravenously. They are not effective against viral infections, and using them inappropriately can lead to antibiotic resistance.

Explain the above in one sentence:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

Exercise: Instruct the model to explain the paragraph in one sentence like "I am 5". Do you see any differences?

In [ ]:
# Your solution here


### 1.2 Question Answering

In [ ]:
params = set_open_params()
prompt = """Answer the question based on the context below. Keep the answer short and concise. Respond "Unsure about answer" if not sure about the answer.

Context: Teplizumab traces its roots to a New Jersey drug company called Ortho Pharmaceutical. There, scientists generated an early version of the antibody, dubbed OKT3. Originally sourced from mice, the molecule was able to bind to the surface of T cells and limit their cell-killing potential. In 1986, it was approved to help prevent organ rejection after kidney transplants, making it the first therapeutic antibody allowed for human use.

Question: What was OKT3 originally sourced from?

Answer:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)


Context obtained from here: https://www.nature.com/articles/d41586-023-00400-x

Exercise: Edit prompt and get the model to respond that it isn't sure about the answer.

In [ ]:
# Your solution here


### 1.3 Text Classification

In [ ]:
prompt = """Classify the text into neutral, negative or positive.

Text: I think the food was okay.

Sentiment:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

Exercise: Modify the prompt to instruct the model to provide an explanation to the answer selected.

In [ ]:
# Your solution here


### 1.4 Role Playing

In [ ]:
prompt = """The following is a conversation with an AI research assistant. The assistant tone is technical and scientific.

Human: Hello, who are you?
AI: Greeting! I am an AI research assistant. How can I help you today?
Human: Can you tell me about the creation of blackholes?
AI:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

Exercise: Modify the prompt to instruct the model to keep AI responses concise and short.

In [ ]:
# Your solution here


### 1.5 Code Generation

In [ ]:
prompt = "\"\"\"\nTable departments, columns = [DepartmentId, DepartmentName]\nTable students, columns = [DepartmentId, StudentId, StudentName]\nCreate a MySQL query for all students in the Computer Science Department\n\"\"\""

response = get_completion(params, prompt)
IPython.display.Markdown(response)


### 1.6 Reasoning

In [ ]:
prompt = """The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1.

Solve by breaking the problem into steps. First, identify the odd numbers, add them, and indicate whether the result is odd or even."""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

Exercise: Improve the prompt to have a better structure and output format.

In [ ]:
# Your solution here


---

## 2. Advanced Prompting Techniques

Objectives:

- Cover more advanced techniques for prompting: few-shot, chain-of-thoughts,...

### 2.2 Few-shot prompts

In [ ]:
prompt = """The odd numbers in this group add up to an even number: 4, 8, 9, 15, 12, 2, 1.
A: The answer is False.

The odd numbers in this group add up to an even number: 17,  10, 19, 4, 8, 12, 24.
A: The answer is True.

The odd numbers in this group add up to an even number: 16,  11, 14, 4, 8, 13, 24.
A: The answer is True.

The odd numbers in this group add up to an even number: 17,  9, 10, 12, 13, 4, 2.
A: The answer is False.

The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1.
A:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

### 2.3 Chain-of-Thought (CoT) Prompting

In [ ]:
prompt = """The odd numbers in this group add up to an even number: 4, 8, 9, 15, 12, 2, 1.
A: Adding all the odd numbers (9, 15, 1) gives 25. The answer is False.

The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1.
A:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

### 2.4 Zero-shot CoT

In [ ]:
prompt = """I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman. I then went and bought 5 more apples and ate 1. How many apples did I remain with?

Let's think step by step."""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

---

## Challenge 1: Advanced Few-Shot Prompting

**Technique:** Provide multiple high-quality exemplars so the model learns your preferred style of solution.

**Input Question:**  
Find the domain of  
*f(x) = √(x² − 5x + 6)*

**Task:**  
Design a prompt with **3** solved examples of “find the domain of a square-root function”

1. States the function.  
2. Explains the inequality step by step.  
3. Gives the final domain in interval notation.

> **Example prompt**  
> ```text
> Example 1:
> Q: Find the domain of f(x) = √(2x − 4).
> A:
>  1. Require 2x − 4 ≥ 0  
>  2. 2x ≥ 4 ⇒ x ≥ 2  
>  3. Domain: [2, ∞)
>
> Example 2:
> Q: Find the domain of f(x) = √(5 − x).
> A:
>  1. Require 5 − x ≥ 0  
>  2. −x ≥ −5 ⇒ x ≤ 5  
>  3. Domain: (−∞, 5]
>
> Example 3:
> Q: Find the domain of f(x) = √(x + 1).
> A:
>  1. Require x + 1 ≥ 0  
>  2. x ≥ −1  
>  3. Domain: [−1, ∞)
>
> **Now you:**
> Q: Find the domain of f(x) = √(3x − 2).
> A:
> ```

In [ ]:
# Your solution here


---

## Challenge 2: Tree-of-Thought Prompting

**Technique:** Encourage the model to **branch** its reasoning, exploring multiple sub-steps in parallel before committing to an answer.

**Input Question:**  
Find the domain of  
*f(x) = √(9 − x²) / (x − 2)*

**Task:**  
Write a prompt that tells the model to:

1. **List** all conditions (branches) that must be satisfied.  
2. **Evaluate** each branch separately.  
3. **Combine** them for the final domain.

> **Example prompt**  
> ```text
> Solve for the domain of f(x) = √(3 − x²) using a Tree-of-Thought:
>
> **Step 1**: List branches  
> - Branch A: 3 − x² ≥ 0  
>
> **Step 2**: Explore each branch  
> > *Branch A1*: Solve 3 − x² ≥ 0  
> > - x² ≤ 3 ⇒ −√3 ≤ x ≤ √3  
>
> **Step 3**: Combine results  
> - Only one branch, so domain is [−√3, √3]
>
> **Answer**: [−√3, √3]
> ```


In [ ]:
# Your solution here


---

## Challenge 3: Program-Aided Language Models

**Technique:** Instruct the model to **call external code** (e.g., Python/Sympy) to handle the algebra, then summarize the result.

**Input Question:**  
Find the domain of  
*f(x) = √(x + 5) / √(10 − x)*

**Task:**  
Prompt the model to:

1. Write a short Python script using Sympy to solve an inequality.  
2. Run it mentally (or via a stub) and then present the domain.

> **Example prompt**  
> ```text
> **Question**: Find the domain of f(x) = √(3x − 2).
>
> **Instructions**:
> 1. Write Python code with Sympy to solve 3x − 2 ≥ 0.  
> 2. Execute the code and capture the output.  
> 3. State the domain in interval notation.
>
> ```python
> import sympy as sp
> x = sp.symbols('x')
> expr = 3*x - 2
> solution = sp.solve_univariate_inequality(expr >= 0, x)
> print(solution)
> ```
>

In [ ]:
# Your solution here
